# Wikipedia RAG Indexing Pipeline (FAISS)

Creates a memory-efficient FAISS index for RAG evaluation with popularity metadata.

## Advantages over Elasticsearch:
- **115x less RAM** with memory-mapped IVF_PQ indices
- Fully local — no external services required
- Faster exact search for smaller datasets
- Easy to snapshot and version (just copy files)
- Works on machines with limited RAM (4GB+)

## Memory Comparison (4M docs, 1536-dim embeddings):
- **Elasticsearch/Flat FAISS**: ~23 GB RAM (full vectors)
- **FAISS HNSW**: ~51 GB RAM (vectors + graph)
- **FAISS IVF_PQ**: ~200 MB RAM (compressed)
- **FAISS IVF_PQ with mmap**: ~50 MB active RAM (115x less!)

## Prerequisites:
```bash
# Install FAISS (CPU version)
pip install -qU faiss-cpu

# OR GPU version (faster for indexing, requires CUDA)
pip install -qU faiss-gpu
```

## Steps:
1. Load QA datasets from HuggingFace
2. Load Wikipedia corpus
3. Add popularity metadata
4. Create FAISS index with memory-efficient settings
5. Save index to disk (with SQLite docstore)
6. Test loading and searching

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from src.rag.faiss_rag_service import FaissRagService, MemoryConfig
from src.rag.utils import IndexingConfig
from config import DATA_DIR, CACHE_DIR
from tqdm import tqdm
import logging
import dotenv
import os
import gc

dotenv.load_dotenv()

# Suppress noisy HTTP logs from libraries
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

# Enable verbose FAISS logging for troubleshooting
logging.getLogger("rag.faiss_rag_service").setLevel(logging.INFO)

# ============================================================================

# ── FAISS Strategy ──────────────────────────────────────────────────────────
# Options: "vector" (exact, high RAM), "hnsw" (fast approx, medium RAM),
#          "ivfpq" (low RAM), "ivfpq_disk" (minimal RAM), "opq_ivfpq" (better accuracy)
STRATEGY = "ivfpq"  # Recommended for large datasets with limited RAM

# ── Datasets ─────────────────────────────────────────────────────────────────
QA_DATASETS = []  # e.g., ["natural_questions", "hotpot_qa"]
WIKIPEDIA_DATASET = "facebook/kilt_wikipedia"
WIKIPEDIA_VERSION = "2019-08-01"
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# ── Index naming & paths ─────────────────────────────────────────────────────
NAME = "wiki_faiss_test"
COLLECTION_NAME = NAME
N_RANDOM_SAMPLES = 10_000  # Set to None to load all docs, 0 for QA-required docs only
COLLECTION_ROOT = Path(DATA_DIR) / NAME
QUESTIONS_PATH = COLLECTION_ROOT / "train_questions.parquet"
WIKI_PARQUET_PATH = COLLECTION_ROOT / "wiki_corpus.parquet"
FAISS_INDEX_PATH = COLLECTION_ROOT / "faiss_index"
QUERY_PROMT_PATH = Path(DATA_DIR) / "prompts" / "query_plain.txt"
PASSAGE_PROMT_PATH = Path(DATA_DIR) / "prompts" / "passage_plain.txt"

# ── Embedding (runtime — no Modal redeploy needed) ──────────────────────────
EMBEDDING_PROVIDER = "modal"  # or "huggingface", "openai"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"  # "Lajavaness/bilingual-embedding-small"
GPU_BATCH_SIZE = 256
REQUEST_BATCH_SIZE = 2048
NORMALISE_EMBEDDINGS = True

# ── Text chunking ────────────────────────────────────────────────────────────
CHUNK_SIZE = 1000                  # max chars per document chunk
CHUNK_OVERLAP = 100                # overlap between chunks

# ── Indexing pipeline ────────────────────────────────────────────────────────
BATCH_SIZE = 10_000  # Lower than ES because FAISS is faster

# ── Memory Configuration ─────────────────────────────────────────────────────
MAX_RAM_MB = 4096  # Approximate RAM budget in MB
USE_MMAP = True    # Memory-map the index (recommended for large indices)
USE_ONDISK_IVF = False  # Store inverted lists on disk (for billion-scale)
TRAINING_SAMPLE_SIZE = 500_000  # Max vectors to sample for IVF training

# ── FAISS Index Parameters ───────────────────────────────────────────────────
# HNSW (if strategy="hnsw")
HNSW_M = 32  # Connections per layer (higher = more memory, better recall)
HNSW_EF_CONSTRUCTION = 200  # Higher = slower build, better quality
HNSW_EF_SEARCH = 128  # Higher = slower search, better recall

# IVF_PQ (if strategy="ivfpq", "ivfpq_disk", or "opq_ivfpq")
IVFPQ_NLIST = 4096  # Number of clusters (√n to 4√n is good)
IVFPQ_M = 48  # Number of PQ subvectors (must divide embedding dim evenly)
IVFPQ_NBITS = 8  # Bits per subvector (8 = 256 centroids per subvector)
IVFPQ_NPROBE = 64  # Clusters to search (higher = slower, better recall)

# OPQ (if strategy="opq_ivfpq")
OPQ_M = 48  # OPQ rotation dimension (usually same as IVFPQ_M)

# ── Balancing & Synthetic ────────────────────────────────────────────────────
BALANCE_DECILES = False
ADD_SYNTHETIC_QUESTIONS = False
MIN_QUESTIONS_PER_DECILE = 200
MODEL_NAME = "gpt-4.1-nano"
SYNTHETIC_BATCH_SIZE = 500

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"✓ Config loaded: {QA_DATASETS} → {COLLECTION_NAME}")
print(f"  Strategy: {STRATEGY} | Embedding: {EMBEDDING_PROVIDER}")
print(f"  Chunk: {CHUNK_SIZE} chars (overlap {CHUNK_OVERLAP})")
print(f"  Batch size: {BATCH_SIZE:,}")
print(f"  RAM budget: {MAX_RAM_MB:,} MB | mmap: {USE_MMAP}")
print(f"  Balance: {BALANCE_DECILES} | Synthetic: {ADD_SYNTHETIC_QUESTIONS}")
if STRATEGY in ["ivfpq", "ivfpq_disk", "opq_ivfpq"]:
    print(f"  IVF_PQ: nlist={IVFPQ_NLIST}, m={IVFPQ_M}, nprobe={IVFPQ_NPROBE}")
elif STRATEGY == "hnsw":
    print(f"  HNSW: M={HNSW_M}, ef_construction={HNSW_EF_CONSTRUCTION}, ef_search={HNSW_EF_SEARCH}")

In [ ]:
# ============================================================================
# STEP 1: Load QA Dataset (prepared by scripts/prepare_qa_dataset.py)
# ============================================================================
# To prepare the QA dataset from scratch, run:
#   python scripts/prepare_qa_dataset.py \
#       --qa-datasets natural_questions triviaqa \
#       --output data/wiki_faiss_test/train_questions.parquet \
#       --balance
# ============================================================================

from scripts.prepare_qa_dataset import (
    prepare_qa_dataset,
)

if QUESTIONS_PATH.exists():
    print(f"Loading existing QA from {QUESTIONS_PATH}...")
    qa_df = pd.read_parquet(QUESTIONS_PATH)
    qa_df["wikipedia_id"] = pd.to_numeric(qa_df["wikipedia_id"], errors="coerce").astype(int)
    print(f"✓ Loaded {len(qa_df):,} questions")
else:
    print("No existing QA found — preparing from HuggingFace...")
    if not QA_DATASETS:
        logging.warning("No QA datasets specified! The index will be created without question-answer pairs, which may affect downstream evaluation.")
        qa_df = pd.DataFrame(columns=["question_text", "answer", "wikipedia_id"])
    else:
        qa_df = prepare_qa_dataset(
            qa_datasets=QA_DATASETS,
            popularity_dataset=POPULARITY_DATASET,
            output_path=QUESTIONS_PATH,
            balance=BALANCE_DECILES,
            cache_dir=CACHE_DIR,
        )

required_doc_ids = set(qa_df["wikipedia_id"])
print(f"\n✓ QA: {len(qa_df):,} questions | Documents needed: {len(required_doc_ids):,}")

if "decile" in qa_df.columns:
    print(f"Distribution:\n{qa_df['decile'].value_counts().sort_index()}")

display(qa_df.head())

In [ ]:
# ============================================================================
# STEP 2: Load Wikipedia (Only Required Documents)
# ============================================================================

if not WIKI_PARQUET_PATH.exists():
    print("\nLoading Wikipedia...")
    # Keep a reference to full dataset for retrieval
    full_wiki_ds = load_dataset(WIKIPEDIA_DATASET, WIKIPEDIA_VERSION, split="full", cache_dir=CACHE_DIR)
    full_wiki_ds = full_wiki_ds.select_columns(["wikipedia_id", "wikipedia_title", "text"])

    if N_RANDOM_SAMPLES is None:
        print("No random sampling configured, loading all documents...")
        wiki_ds = full_wiki_ds
    elif len(required_doc_ids) < N_RANDOM_SAMPLES:
        print(f"Sampling {N_RANDOM_SAMPLES:,} random documents from Wikipedia...")
        wiki_ds = full_wiki_ds.shuffle(seed=42).select(range(N_RANDOM_SAMPLES))
        print(f"✓ Loaded {len(wiki_ds):,} articles")
        
        # Check for missing required documents
        print("Checking for missing required documents...")
        existing_ids = set(int(i) for i in wiki_ds["wikipedia_id"] if i is not None)
        missing_ids = required_doc_ids - existing_ids
        print(f"✓ Missing: {len(missing_ids):,}")
        
        # Add missing docs if needed (OPTIMIZED - batched filtering)
        if missing_ids:
            print("  Fetching missing documents (this may take a moment)...")
            missing_ds = full_wiki_ds.filter(
                lambda batch: [int(i) in missing_ids for i in batch["wikipedia_id"]],
                batched=True,
                batch_size=10000,
                desc="Finding missing docs"
            )
            wiki_ds = concatenate_datasets([wiki_ds, missing_ds])
            print(f"✓ Added {len(missing_ds):,} missing docs. Total: {len(wiki_ds):,}")
    elif N_RANDOM_SAMPLES == 0:
        # Load only required documents (OPTIMIZED - batched filtering)
        print(f"Loading only {len(required_doc_ids):,} required documents...")
        if required_doc_ids:
            wiki_ds = full_wiki_ds.filter(
                lambda batch: [int(i) in required_doc_ids for i in batch["wikipedia_id"]],
                batched=True,
                batch_size=10000,
                desc="Loading required docs"
            )
            print(f"✓ Loaded {len(wiki_ds):,} articles")
        else:
            print("⚠️  No documents required, using empty dataset")
            wiki_ds = full_wiki_ds.select([])

    # Flatten KILT text structure - ALWAYS CHECK
    print("Normalizing text format...")

    def flatten_text(batch):
        return {
            "text": [
                "\n".join(t["paragraph"]) if isinstance(t, dict) and "paragraph" in t else str(t)
                for t in batch["text"]
            ]
        }

    # Apply to first item to check if needed
    needs_flattening = False
    if len(wiki_ds) > 0:
        sample_text = wiki_ds[0]["text"]
        if isinstance(sample_text, dict):
            needs_flattening = True

    if needs_flattening:
        wiki_ds = wiki_ds.map(flatten_text, batched=True, desc="Flattening text")
        print("✓ Text flattened")
    else:
        print("✓ Text already flat (or empty)")

In [ ]:
# ============================================================================
# STEP 3: Add Popularity Metadata
# ============================================================================

if WIKI_PARQUET_PATH.exists():
    print("Wikipedia Parquet already exists, loading directly...")
else:
    print("Loading popularity data...")
    pop_ds = load_dataset(POPULARITY_DATASET, split="train+test", cache_dir=CACHE_DIR)

    # Detect columns
    cols = pop_ds.column_names
    id_col = "wikipedia_id" if "wikipedia_id" in cols else "id"
    rank_col = next((c for c in ["rank_avg", "avg_rank"] if c in cols), None)

    # Get needed IDs early
    needed_ids = set(int(x) for x in wiki_ds["wikipedia_id"] if x is not None)
    print(f"✓ Target documents: {len(needed_ids):,}")

    print("Processing popularity metadata (OPTIMIZED - Vectorized)...")

    # STEP 1: Load MINIMAL data first
    print("  Loading popularity scores...")
    pop_df_minimal = pop_ds.select_columns([id_col, "popularity_avg"]).to_pandas()
    pop_df_minimal[id_col] = pd.to_numeric(pop_df_minimal[id_col], errors='coerce').fillna(-1).astype(int)

    # STEP 2: FILTER EARLY - Keep only needed rows
    print(f"  Filtering from {len(pop_df_minimal):,} to {len(needed_ids):,} rows...")
    relevant_pop_df = pop_df_minimal[pop_df_minimal[id_col].isin(needed_ids)].copy()

    # STEP 3: Add rank column if needed
    if rank_col and rank_col in pop_ds.column_names:
        print("  Adding rank data for relevant documents only...")
        pop_ds_filtered = pop_ds.filter(
            lambda batch: [int(i) in needed_ids for i in batch[id_col]],
            batched=True,
            batch_size=10000,
            desc="Filtering popularity data",
        )
        rank_df = pop_ds_filtered.select_columns([id_col, rank_col]).to_pandas()
        rank_df[id_col] = rank_df[id_col].astype(int)
        relevant_pop_df = relevant_pop_df.merge(rank_df, on=id_col, how="left")

    # Standardize rank column name
    if rank_col:
        relevant_pop_df = relevant_pop_df.rename(columns={rank_col: "popularity_rank"})
    else:
        relevant_pop_df["popularity_rank"] = None

    # STEP 4: Build lookup dictionary (NO decile — boundaries are computed downstream)
    print("  Building optimized lookup...")
    pop_lookup = relevant_pop_df.set_index(id_col).to_dict(orient="index")

    # Cleanup
    del pop_df_minimal, pop_ds
    if rank_col:
        del pop_ds_filtered, rank_df
    gc.collect()

    print(f"✓ Built lookup with {len(pop_lookup):,} entries")

    # Merge with Wikipedia using batched map
    print("Merging metadata...")

    def merge_batch(batch):
        """Vectorized metadata merge — popularity only, no decile"""
        ids = [int(i) for i in batch["wikipedia_id"]]
        defaults = {"popularity_avg": None, "popularity_rank": None}
        meta_list = [pop_lookup.get(i, defaults) for i in ids]

        return {
            "popularity_avg": [m.get("popularity_avg") for m in meta_list],
            "popularity_rank": [m.get("popularity_rank") for m in meta_list],
        }

    wiki_ds_with_pop = wiki_ds.map(
        merge_batch,
        batched=True,
        batch_size=10_000,
        desc="Merging",
    )

    print(f"✓ Ready for indexing: {len(wiki_ds_with_pop):,} documents")

In [ ]:
# ============================================================================
# STEP 3.5: (Optional) Synthetic Question Generation
# ============================================================================
# Handled by: python scripts/prepare_qa_dataset.py --generate-synthetic --corpus ...
# This cell is a no-op unless you want to re-generate inline.
# ============================================================================

if ADD_SYNTHETIC_QUESTIONS:
    from scripts.prepare_qa_dataset import generate_synthetic

    print("\n🤖 Generating synthetic questions...")
    qa_df = generate_synthetic(
        qa_df,
        corpus_path=WIKI_PARQUET_PATH,
        questions_per_decile=MIN_QUESTIONS_PER_DECILE,
        model_name=MODEL_NAME,
        batch_size=SYNTHETIC_BATCH_SIZE,
    )
    if BALANCE_DECILES:
        from scripts.prepare_qa_dataset import balance_by_decile
        qa_df = balance_by_decile(qa_df)
    print(f"✓ QA updated: {len(qa_df):,} questions")
else:
    print("✓ Skipping synthetic generation (ADD_SYNTHETIC_QUESTIONS = False)")

In [ ]:
# ===========================================================================
# STEP 3.7: Data Sanitization & Save to Parquet (streaming to avoid RAM)
# ===========================================================================

print("\n🧹 Sanitizing data for FAISS indexing (streaming)...")
COLLECTION_ROOT.mkdir(parents=True, exist_ok=True)

print(f"\n💾 Saving documents to {WIKI_PARQUET_PATH}...")
if WIKI_PARQUET_PATH.exists():
    print(f"⚠️  Warning: {WIKI_PARQUET_PATH} already exists and existing will be used.")
else:
    writer = None
    n_docs = 0
    batch_size = 50_000

    # Calculate total batches for progress bar
    total_docs = len(wiki_ds_with_pop)
    total_batches = (total_docs + batch_size - 1) // batch_size

    pbar = tqdm(total=total_batches, desc="Saving to parquet", unit="batch")

    for batch in wiki_ds_with_pop.iter(batch_size=batch_size):
        batch_df = pd.DataFrame(batch)

        # Ensure numeric conversions happen after filling NaNs to avoid IntCastingNaNError
        batch_df["wikipedia_id"] = pd.to_numeric(batch_df["wikipedia_id"], errors="coerce").astype(int)

        # Use pandas nullable Float64 dtype for columns that may contain NaN
        batch_df["popularity_avg"] = pd.to_numeric(batch_df["popularity_avg"], errors="coerce").fillna(-1).astype("Float64")
        batch_df["popularity_rank"] = pd.to_numeric(batch_df["popularity_rank"], errors="coerce").astype("Float64")

        table = pa.Table.from_pandas(batch_df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(WIKI_PARQUET_PATH, table.schema)
        writer.write_table(table)

        n_docs += len(batch_df)
        pbar.update(1)
        
        del batch_df, table
        gc.collect()

    pbar.close()

    if writer is not None:
        writer.close()

    print(f"  ✓ Saved {n_docs:,} documents ({WIKI_PARQUET_PATH.stat().st_size / 1e9:.2f} GB)")
    del wiki_ds_with_pop, wiki_ds, full_wiki_ds
    gc.collect()


# ── Save training questions ──────────────────────────────────────────────────
print(f"Saving training questions to {QUESTIONS_PATH}...")
if QUESTIONS_PATH.exists():
    print(f"⚠️  Warning: {QUESTIONS_PATH} already exists and existing will be used.")
else:
    qa_df.to_parquet(QUESTIONS_PATH, index=False, engine="pyarrow")
    print(f"  ✓ Saved {len(qa_df):,} questions")

# ── Free memory — everything is on disk now ──────────────────────────────────
del qa_df
gc.collect()

In [ ]:
# ============================================================================
# STEP 4: Create FAISS Index (Memory-Efficient Streaming)
# ============================================================================

print("\n📊 Creating FAISS index (streaming from Parquet)...")

# ── Build IndexingConfig ─────────────────────────────────────────────────────
config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider=EMBEDDING_PROVIDER,
    embedding_model=EMBEDDING_MODEL,
    gpu_batch_size=GPU_BATCH_SIZE,
    normalise_embeddings=NORMALISE_EMBEDDINGS,
    request_batch_size=REQUEST_BATCH_SIZE,
    passage_prompt_file=str(PASSAGE_PROMT_PATH) if PASSAGE_PROMT_PATH.exists() else None,
    query_prompt_file=str(QUERY_PROMT_PATH) if QUERY_PROMT_PATH.exists() else None,
    trust_remote_code=True,
    use_progress=False,
)

# ── Build MemoryConfig ───────────────────────────────────────────────────────
memory_config = MemoryConfig(
    max_ram_mb=MAX_RAM_MB,
    use_mmap=USE_MMAP,
    use_ondisk_ivf=USE_ONDISK_IVF,
    training_sample_size=TRAINING_SAMPLE_SIZE,
    queue_maxsize=2,
    gc_every_n_batches=1,
    prefetch_batches=1,
)

# ── Initialize FAISS Service ─────────────────────────────────────────────────
service = FaissRagService(
    config=config,
    strategy=STRATEGY,
    distance_strategy="cosine",
    memory_config=memory_config,
    # HNSW parameters (only used if strategy="hnsw")
    hnsw_m=HNSW_M,
    hnsw_ef_construction=HNSW_EF_CONSTRUCTION,
    hnsw_ef_search=HNSW_EF_SEARCH,
    # IVF_PQ parameters (only used if strategy in ["ivfpq", "ivfpq_disk", "opq_ivfpq"])
    ivfpq_nlist=IVFPQ_NLIST,
    ivfpq_m=IVFPQ_M,
    ivfpq_nbits=IVFPQ_NBITS,
    ivfpq_nprobe=IVFPQ_NPROBE,
    # OPQ parameters (only used if strategy="opq_ivfpq")
    opq_m=OPQ_M,
)

print(f"  Strategy: {STRATEGY}")
print(f"  Config: {config}")
print(f"  Memory: {memory_config}")

# ── Index from Parquet ───────────────────────────────────────────────────────
print(f"\n📥 Indexing from {WIKI_PARQUET_PATH}...")

faiss_store, num_chunks = service.index_from_parquet_batches(
    parquet_path=WIKI_PARQUET_PATH,
    text_field="text",
    metadata_fields=["wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"],
    batch_size=BATCH_SIZE,
    progress_bar=True,
)

print(f"\n✅ Indexing complete!")
print(f"  Total chunks indexed: {num_chunks:,}")

# ── Get index statistics ─────────────────────────────────────────────────────
stats = service.get_index_stats()
print(f"\n📊 Index Statistics:")
for key, value in stats.items():
    if isinstance(value, int):
        print(f"  {key}: {value:,}")
    elif isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

# ── Estimate memory usage ────────────────────────────────────────────────────
mem_estimate = service.estimate_index_memory()
print(f"\n💾 Estimated memory usage: {mem_estimate / 1024**2:.2f} MB")

In [ ]:
# ============================================================================
# STEP 5: Save FAISS Index to Disk
# ============================================================================

print(f"\n💾 Saving FAISS index to {FAISS_INDEX_PATH}...")

if FAISS_INDEX_PATH.exists():
    print(f"⚠️  Warning: {FAISS_INDEX_PATH} already exists. Overwriting...")
    import shutil
    shutil.rmtree(FAISS_INDEX_PATH)

service.save_faiss_store(FAISS_INDEX_PATH)

print(f"✅ Index saved successfully!")
print(f"  Location: {FAISS_INDEX_PATH}")
print(f"  Size: {sum(f.stat().st_size for f in FAISS_INDEX_PATH.rglob('*') if f.is_file()) / 1024**2:.2f} MB")

# List all files in the index directory
print(f"\n📁 Index files:")
for f in sorted(FAISS_INDEX_PATH.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.relative_to(FAISS_INDEX_PATH)}: {size_mb:.2f} MB")

In [ ]:
# ============================================================================
# STEP 6: Test Loading and Searching
# ============================================================================

print("\n🔍 Testing index loading and search...")

# Clear current service to test loading from disk
del service, faiss_store
gc.collect()

# Create a fresh service and load the index
test_service = FaissRagService(
    config=config,
    strategy=STRATEGY,
    distance_strategy="cosine",
    memory_config=memory_config,
    ivfpq_nprobe=IVFPQ_NPROBE,
)

print(f"Loading index from {FAISS_INDEX_PATH}...")
test_service.load_faiss_store(FAISS_INDEX_PATH, use_mmap=USE_MMAP)
print("✅ Index loaded successfully!")

# Test search
test_queries = [
    "What is machine learning?",
    "Who invented the telephone?",
    "What is the capital of France?",
]

print(f"\n🔎 Testing retrieval with {len(test_queries)} queries...")
for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}: {query}")
    results = test_service.retrieve_documents(query, k=3)
    print(f"  Retrieved {len(results)} documents:")
    for j, doc in enumerate(results, 1):
        title = doc.metadata.get("wikipedia_title", "Unknown")
        wiki_id = doc.metadata.get("wikipedia_id", "Unknown")
        pop_avg = doc.metadata.get("popularity_avg", "Unknown")
        snippet = doc.page_content[:100].replace("\n", " ")
        print(f"    {j}. {title} (ID: {wiki_id}, pop: {pop_avg})")
        print(f"       {snippet}...")

print("\n✅ All tests passed!")
print(f"\n📌 Your FAISS index is ready at: {FAISS_INDEX_PATH}")
print(f"   Load it with: service.load_faiss_store('{FAISS_INDEX_PATH}', use_mmap={USE_MMAP})")

In [ ]:
# ============================================================================
# (Optional) Compare Memory Usage: mmap vs. full load
# ============================================================================

import psutil
import os

def get_memory_usage_mb():
    """Get current process memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2

print("\n📊 Memory Usage Comparison")
print("=" * 60)

# Test 1: Load with mmap
del test_service
gc.collect()
mem_before = get_memory_usage_mb()

service_mmap = FaissRagService(
    config=config,
    strategy=STRATEGY,
    distance_strategy="cosine",
    memory_config=memory_config,
)
service_mmap.load_faiss_store(FAISS_INDEX_PATH, use_mmap=True)
mem_after_mmap = get_memory_usage_mb()

print(f"Memory-mapped loading:")
print(f"  Before: {mem_before:.2f} MB")
print(f"  After:  {mem_after_mmap:.2f} MB")
print(f"  Delta:  {mem_after_mmap - mem_before:.2f} MB")

# Test 2: Load without mmap (full load into RAM)
del service_mmap
gc.collect()
mem_before = get_memory_usage_mb()

service_full = FaissRagService(
    config=config,
    strategy=STRATEGY,
    distance_strategy="cosine",
    memory_config=MemoryConfig(use_mmap=False),
)
service_full.load_faiss_store(FAISS_INDEX_PATH, use_mmap=False)
mem_after_full = get_memory_usage_mb()

print(f"\nFull RAM loading:")
print(f"  Before: {mem_before:.2f} MB")
print(f"  After:  {mem_after_full:.2f} MB")
print(f"  Delta:  {mem_after_full - mem_before:.2f} MB")

ratio = (mem_after_full - mem_before) / (mem_after_mmap - mem_before) if (mem_after_mmap - mem_before) > 0 else 0
print(f"\n💡 Memory-mapped loading uses {ratio:.1f}x less RAM!")

del service_full
gc.collect()